# Online Learning & Regret

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/streaming-ml/03-online-learning

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

**The problem.** Fit a model to a stream one example at a time, one pass, no stored dataset — even when the data isn't i.i.d. **The core idea:** **online gradient descent** takes one gradient step per example, and its **regret** (loss versus the best fixed decision in hindsight) grows only like `√T` — so average regret → 0. This is also *why* SGD works. We build OGD from scratch and check it against scikit-learn's `SGDClassifier`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'
np.random.seed(42)

## 1. From scratch — OGD on convex losses, and its regret

Each round the environment reveals the convex loss `f_t(x) = |x - y_t|`; OGD takes a subgradient step `x_{t+1} = x_t - η_t · sign(x_t - y_t)` with `η_t = c/√t`. **Regret** compares OGD's cumulative loss to the single **best fixed decision in hindsight** — for absolute loss that is the *median* of `y`. We contrast OGD against a learner that plays a fixed, poor guess and never adapts.

In [ ]:
T = 3000
y = np.random.randn(T)                 # stationary i.i.d. targets
x_star = np.median(y)                  # best fixed decision in hindsight (abs loss -> median)

def run_ogd(y, c):
    x, loss, comp = 0.0, 0.0, 0.0
    regret = np.empty(len(y))
    for t, yt in enumerate(y, start=1):
        loss += abs(x - yt)
        comp += abs(x_star - yt)
        regret[t - 1] = loss - comp
        x -= (c / np.sqrt(t)) * np.sign(x - yt)   # subgradient step
    return regret

def run_fixed(y, x0):
    loss, comp = 0.0, 0.0
    regret = np.empty(len(y))
    for t, yt in enumerate(y, start=1):
        loss += abs(x0 - yt)
        comp += abs(x_star - yt)
        regret[t - 1] = loss - comp
    return regret

regret_ogd = run_ogd(y, c=1.0)
regret_lazy = run_fixed(y, x0=2.5)      # a fixed, poor decision -> linear regret
assert regret_ogd[-1] >= 0, 'static regret vs the best fixed point is non-negative'
print(f'OGD    regret R_T = {regret_ogd[-1]:.1f}   avg R_T/T = {regret_ogd[-1]/T:.4f}')
print(f'fixed  regret R_T = {regret_lazy[-1]:.1f}   avg R_T/T = {regret_lazy[-1]/T:.4f}')

## 2. The library way — cross-check against scikit-learn SGD

`SGDClassifier` with `partial_fit` **is** online gradient descent on the hinge/log loss. We stream a linearly-separable problem through our own one-pass logistic learner and through sklearn's SGD, and assert both reach similar accuracy.

In [ ]:
from sklearn.linear_model import SGDClassifier

d = 6
w_true = np.random.randn(d)
def gen(nn):
    X = np.random.randn(nn, d)
    p = 1 / (1 + np.exp(-X @ w_true))
    return X, (np.random.rand(nn) < p).astype(int)

Xtr, ytr = gen(6000); Xte, yte = gen(2000)

# our from-scratch online logistic regression (predict-then-update)
w = np.zeros(d)
for t in range(len(Xtr)):
    x = Xtr[t]
    p = 1 / (1 + np.exp(-w @ x))
    w -= (0.5 / np.sqrt(t + 1)) * (p - ytr[t]) * x
ours = np.mean((1 / (1 + np.exp(-Xte @ w)) > 0.5) == yte)

# sklearn SGD, streamed one mini-batch at a time via partial_fit
sgd = SGDClassifier(loss='log_loss', learning_rate='optimal', random_state=0)
for i in range(0, len(Xtr), 100):
    sgd.partial_fit(Xtr[i:i+100], ytr[i:i+100], classes=[0, 1])
sk = sgd.score(Xte, yte)

print(f'our online logistic accuracy = {ours:.3f}   sklearn SGD accuracy = {sk:.3f}')
assert abs(ours - sk) < 0.05, 'our OGD-style learner must match sklearn SGD'
print('from-scratch online GD == sklearn SGDClassifier (same ballpark) ✓')

## 3. Visualize it — regret grows like √T

Plot cumulative regret for OGD and for the fixed poor-guess learner, against a `√T` reference.

In [ ]:
ts = np.arange(1, T + 1)
ref = regret_ogd[-1] / np.sqrt(T) * np.sqrt(ts)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ts, regret_lazy, color='#f43f5e', label='fixed guess  (~T)')
ax.plot(ts, regret_ogd, color='#14b8a6', label='OGD  (~√T)')
ax.plot(ts, ref, '--', color='#e2e8f0', label='√T reference')
ax.set_xlabel('rounds T'); ax.set_ylabel('cumulative regret R_T')
ax.set_title('Sublinear regret: OGD tracks √T', color='white')
ax.legend(); ax.grid(alpha=0.2); plt.show()

**What to notice:** OGD's regret hugs the `√T` curve — sublinear, so `R_T/T → 0` and the learner matches the best fixed decision *per round*. The fixed learner's regret is a straight line (`Θ(T)`): a constant penalty every round because it never adapts toward the optimum.

## 4. Tradeoffs & when to use it

- **Step size.** `η_t = c/√t` gives `O(√T)` regret for convex losses; **strongly convex** losses with `η_t = 1/(λt)` improve this to `O(log T)`.
- **No i.i.d. assumption.** Regret is worst-case, so OGD keeps working on drifting/adversarial streams — exactly the streaming setting.
- **Online-to-batch.** Averaging the iterates converts `O(√T)` regret into `O(1/√T)` risk — this is why SGD converges.
- **Costs.** Purely online learners can be noisy and sensitive to feature scaling and step size; mini-batching and per-coordinate adaptivity (AdaGrad/FTRL) stabilise them.

## 5. Your turn

### Exercise 1 — One OGD step on the logistic loss

The gradient of the logistic loss for weights `w`, example `x`, label `y ∈ {0,1}` is `(σ(w·x) - y) · x`. Implement a single decaying-step update returning the new weights.

In [ ]:
def ogd_step(w, x, y, t, c=0.5):
    """One online gradient-descent step; η_t = c/√t (t is 1-indexed)."""
    w = np.asarray(w, dtype=float); x = np.asarray(x, dtype=float)
    # TODO(you): p = sigmoid(w·x); grad = (p - y) * x; return w - (c/sqrt(t)) * grad
    return ...


In [ ]:
# Checks — run me
w0 = np.zeros(3)
x0 = np.array([1.0, 2.0, -1.0])
w1 = ogd_step(w0, x0, y=1, t=1, c=0.5)
# at w=0, sigmoid=0.5, grad = (0.5 - 1)*x = -0.5*x, step = -0.5*(-0.5*x) = 0.25*x
assert np.allclose(w1, 0.25 * x0), 'first step should move weights toward a positive example'
# a correct confident prediction should barely move the weights
w_conf = np.array([10.0, 0.0, 0.0])
assert np.linalg.norm(ogd_step(w_conf, np.array([1.0, 0, 0]), y=1, t=100) - w_conf) < 1e-3
print('✅ Exercise 1 passed')

<details>
<summary>💡 Show solution</summary>

```python
def ogd_step(w, x, y, t, c=0.5):
    w = np.asarray(w, dtype=float); x = np.asarray(x, dtype=float)
    p = 1 / (1 + np.exp(-w @ x))
    grad = (p - y) * x
    return w - (c / np.sqrt(t)) * grad
```

</details>

## 6. Key takeaways

- Online learning is a round-by-round game measured by **regret** vs the best fixed decision.
- **OGD**: `O(√T)` regret (convex), `O(log T)` (strongly convex); average regret → 0.
- **Online-to-batch** turns regret into generalization — SGD is OGD on shuffled data.
- Next: [Concept Drift & Adaptation](https://ml-viz-ruby.vercel.app/courses/streaming-ml/04-concept-drift) — when the stream itself changes.